In [ ]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report,roc_auc_score,roc_curve,auc
import csv
import pandas as pd
from sklearn.feature_extraction.text import TfidfTransformer,TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.naive_bayes import MultinomialNB
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:

df = pd.read_csv(
    r"C:\Users\bharath simha reddy\Desktop\genai_journey\ML\Svm\Data\spam.csv",
    encoding='latin-1'
)

print(df.head())

In [ ]:
df=df[['v1','v2']]
df.columns=['label','message']

In [ ]:
df['label'] = df['label'].map({
    'ham': 0,
    'spam': 1
})

In [ ]:
tf=TfidfVectorizer(stop_words='english')
X=tf.fit_transform(df['message'])
y=df['label']

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42,test_size=0.2)
model=SVC(kernel='linear')
model.fit(X_train,y_train)
y_pred=model.predict(X_test)
print(f"Accuracy:\n{accuracy_score(y_test,y_pred)}")
print(f"classfication Report:\n{classification_report(y_test,y_pred)}")
print(f"Confusion Matrix:\n{confusion_matrix(y_test,y_pred)}")

In [ ]:

# Hyperparameter tuning for SVM
params = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.1, 0.01, 0.001],
    'kernel': ['linear', 'rbf']
}

grid = GridSearchCV(
    SVC(),
    params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,      
    verbose=1
)

grid.fit(X_train, y_train)

# Best SVM model
svm_best = grid.best_estimator_

y_pred_svm = svm_best.predict(X_test)

print("Best SVM Params:", grid.best_params_)
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print("\nSVM Classification Report:\n")
print(classification_report(y_test, y_pred_svm))

print("\nSVM Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_svm))


# Naive Bayes
nb = MultinomialNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

print("\nNaive Bayes Accuracy:",
      accuracy_score(y_test, y_pred_nb))

print("\nNaive Bayes Classification Report:\n")
print(classification_report(y_test, y_pred_nb))

print("\nNaive Bayes Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_nb))

In [ ]:
svm_auc=roc_auc_score(y_test,y_pred_svm)
nb_auc=roc_auc_score(y_test,y_pred_nb)
print('Svm-auc:',svm_auc)
print('nb-auc:',nb_auc)

In [ ]:
cm=confusion_matrix(y_test,y_pred_svm)
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Svm Confusion matrix')
plt.show()

In [ ]:
fpr_svm, tpr_svm, _ = roc_curve(y_test, y_pred_svm)
fpr_nb, tpr_nb, _ = roc_curve(y_test, y_pred_nb)

auc_svm = auc(fpr_svm, tpr_svm)
auc_nb = auc(fpr_nb, tpr_nb)

plt.figure(figsize=(8, 6))
plt.plot(fpr_svm, tpr_svm, label=f'SVM (AUC = {auc_svm:.3f})')
plt.plot(fpr_nb, tpr_nb, label=f'Naive Bayes (AUC = {auc_nb:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for SVM and Naive Bayes')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

In [ ]:
import os
import seaborn as sns

import matplotlib.pyplot as plt

image_dir = r"C:\Users\bharath simha reddy\Desktop\genai_journey\ML\Svm\images"
os.makedirs(image_dir, exist_ok=True)

# Save confusion matrix as PNG
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('SVM Confusion Matrix')
plt.tight_layout()
plt.savefig(os.path.join(image_dir, 'confusion_matrix.png'), dpi=300)
plt.close()

# Save ROC curve plot as workflow.png
fpr_svm, tpr_svm, _ = roc_curve(y_test, y_pred_svm)
fpr_nb, tpr_nb, _ = roc_curve(y_test, y_pred_nb)

auc_svm = auc(fpr_svm, tpr_svm)
auc_nb = auc(fpr_nb, tpr_nb)

plt.figure(figsize=(8, 6))
plt.plot(fpr_svm, tpr_svm, label=f'SVM (AUC = {auc_svm:.3f})')
plt.plot(fpr_nb, tpr_nb, label=f'Naive Bayes (AUC = {auc_nb:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for SVM and Naive Bayes')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(image_dir, 'workflow.png'), dpi=300)
plt.close()

print(f"Images saved to: {image_dir}")